# 02 — Official Concession Acquisition with Playwright
## WIUP ESDM + PBPH Kementerian Kehutanan — Kalimantan

Notebook ini **khusus** mengambil dua layer konsesi resmi:

1. **WIUP pertambangan** dari jalur publik resmi MOMI / ESDM One Map.
2. **PBPH kehutanan** dari SIGAP / Geoportal Kementerian Kehutanan.

Notebook ini **tidak memakai dataset proxy/alternatif**.  
Kalau layer resmi tidak dapat diakses secara anonim, notebook akan **berhenti dan melaporkan status akses**, bukan diam-diam mengganti sumber.

### Strategi
- Playwright membuka aplikasi publik resmi seperti pengguna biasa.
- Browser merekam traffic/config ArcGIS yang benar-benar dipakai aplikasi.
- Notebook mencari service/layer polygon berdasarkan:
  - nama layer,
  - field resmi,
  - tipe geometri.
- Query dilakukan dari **browser session yang sama**.
- Jika server meminta autentikasi/token yang tidak diberikan kepada pengguna publik, notebook tidak mencoba membypass.

### Output
```text
official_concessions/
├── wiup_kalimantan_official.geojson
├── pbph_kalimantan_official.geojson
├── wiup_discovery.json
├── pbph_discovery.json
└── acquisition_manifest.json
```

> Spatial intersection hanya menunjukkan hotspot berada di dalam suatu wilayah izin. Itu bukan bukti bahwa pemegang izin menyebabkan kebakaran.

## 1. Install dependency

In [1]:
# Skipped outside Google Colab.

In [2]:
# Skipped outside Google Colab.

## 2. Konfigurasi

In [3]:
# Skipped outside Google Colab.

In [4]:
from pathlib import Path
import json
import re
import time
import pandas as pd
import geopandas as gpd

from playwright.async_api import async_playwright


# ============================================================
# UBAH PATH INI
# ============================================================

PROJECT_ROOT = Path(
    '\\\\wsl.localhost\\Ubuntu\\home\\adit\\projects\\compfest_academy_2\\data'
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "official_concessions"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# OFFICIAL ENTRY POINTS
# ============================================================

# MOMI public sekarang dapat me-redirect ke public ESDM One Map.
WIUP_ENTRY_URL = (
    "https://momi.minerba.esdm.go.id/public/"
)

# Kita mulai dari SIGAP resmi lalu mencari link Peta Interaktif
# yang sedang aktif, supaya tidak hardcode URL geoportal lama.
SIGAP_ENTRY_URL = (
    "https://sigap.kehutanan.go.id/"
)


# ============================================================
# KALIMANTAN
# ============================================================

KALIMANTAN_BBOX = {
    "xmin": 108.5,
    "ymin": -4.6,
    "xmax": 119.5,
    "ymax": 4.6,
    "spatialReference": {
        "wkid": 4326
    }
}

KALIMANTAN_PROV_NAMES = [
    "Kalimantan Barat",
    "Kalimantan Tengah",
    "Kalimantan Selatan",
    "Kalimantan Timur",
    "Kalimantan Utara",
]

KALIMANTAN_PROV_CODES = [
    61, 62, 63, 64, 65
]


# ============================================================
# OUTPUTS
# ============================================================

WIUP_OUT = (
    OUTPUT_DIR /
    "wiup_kalimantan_official.geojson"
)

PBPH_OUT = (
    OUTPUT_DIR /
    "pbph_kalimantan_official.geojson"
)

WIUP_DISCOVERY_OUT = (
    OUTPUT_DIR /
    "wiup_discovery.json"
)

PBPH_DISCOVERY_OUT = (
    OUTPUT_DIR /
    "pbph_discovery.json"
)

## 3. Helper Playwright + ArcGIS discovery

In [5]:
ARC_SERVICE_RE = re.compile(
    r"https?://[^\"'\s<>]+?"
    r"(?:MapServer|FeatureServer)"
    r"(?:/\d+)?",
    re.I
)

ITEM_ID_RE = re.compile(
    r"\b[a-fA-F0-9]{32}\b"
)


def clean_url(url):
    """
    Buang query string agar token/session parameter
    tidak tercetak atau disimpan ke report.
    """
    if not isinstance(url, str):
        return url

    return url.split("?")[0]


def safe_response_json(response):
    try:
        return response.json()
    except Exception:
        return None


async def request_json_async(
    request_context,
    url,
    params=None,
    timeout=120_000
):
    response = await request_context.get(
        url,
        params=params or {},
        timeout=timeout
    )

    status = response.status

    try:
        data = await response.json()
    except Exception:
        data = {
            "_http_status": status,
            "_url": clean_url(url),
            "_not_json": True
        }

    if isinstance(data, dict):
        data.setdefault(
            "_http_status",
            status
        )

    return data


async def post_json_async(
    request_context,
    url,
    form,
    timeout=180_000
):
    response = await request_context.post(
        url,
        form=form,
        timeout=timeout
    )

    status = response.status

    try:
        data = await response.json()
    except Exception:

        try:
            text = await response.text()
        except Exception:
            text = ""

        data = {
            "_http_status": status,
            "_url": clean_url(url),
            "_text": text[:500]
        }

    if isinstance(data, dict):
        data.setdefault(
            "_http_status",
            status
        )

    return data


def extract_service_urls(obj):
    """
    Recursive scan atas config JSON/text untuk MapServer/FeatureServer.
    """
    found = set()

    def walk(x):

        if isinstance(x, dict):

            for value in x.values():
                walk(value)

        elif isinstance(x, list):

            for value in x:
                walk(value)

        elif isinstance(x, str):

            for match in ARC_SERVICE_RE.findall(x):
                found.add(
                    clean_url(match)
                )

    walk(obj)

    return found


def extract_item_ids(obj):
    found = set()

    def walk(x):

        if isinstance(x, dict):

            for value in x.values():
                walk(value)

        elif isinstance(x, list):

            for value in x:
                walk(value)

        elif isinstance(x, str):

            found.update(
                ITEM_ID_RE.findall(x)
            )

    walk(obj)

    return found


def service_root(url):
    """
    https://.../MapServer/3 -> https://.../MapServer
    """
    return re.sub(
        r"/(?:\d+)$",
        "",
        clean_url(url)
    )


def explicit_layer_id(url):
    match = re.search(
        r"/(?:MapServer|FeatureServer)/(\d+)$",
        clean_url(url),
        re.I
    )

    return (
        int(match.group(1))
        if match
        else None
    )


def auth_error(data):
    if not isinstance(data, dict):
        return False

    err = data.get("error")

    if not isinstance(err, dict):
        return False

    return err.get("code") in {
        401, 403, 498, 499
    }


def arcgis_error(data):
    if not isinstance(data, dict):
        return None

    return data.get("error")


def save_feature_collection(
    features,
    output_path,
    name
):
    fc = {
        "type": "FeatureCollection",
        "name": name,
        "features": features
    }

    output_path.write_text(
        json.dumps(
            fc,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"Saved {len(features):,} features ->",
        output_path
    )

## 4. Browser traffic collector

Collector ini menyimpan:
- final URL setelah redirect,
- resource URL,
- ArcGIS service URL,
- ArcGIS item ID,
- sebagian JSON response yang mengandung kata kunci relevan.

Token/query string tidak disimpan di report.

In [6]:
async def collect_app_traffic_async(
    page,
    entry_url,
    wait_ms=30_000,
    interaction_texts=None
):
    resource_urls = set()

    def on_request(request):

        resource_urls.add(
            clean_url(
                request.url
            )
        )

    page.on(
        "request",
        on_request
    )

    print(
        "Opening:",
        entry_url
    )

    await page.goto(
        entry_url,
        wait_until="domcontentloaded",
        timeout=180_000
    )

    await page.wait_for_timeout(
        wait_ms
    )

    print(
        "Final URL:",
        page.url
    )

    print(
        "Title:",
        await page.title()
    )


    # ========================================================
    # COBA KLIK MENU/LAYER
    # ========================================================

    for text in (
        interaction_texts or []
    ):

        clicked = False

        locators = [
            page.get_by_text(
                text,
                exact=False
            ),
            page.get_by_role(
                "button",
                name=re.compile(
                    re.escape(text),
                    re.I
                )
            )
        ]

        for locator in locators:

            try:
                count = await locator.count()
            except Exception:
                count = 0

            for i in range(
                min(count, 5)
            ):

                try:

                    target = locator.nth(i)

                    if await target.is_visible():

                        await target.click(
                            timeout=3000
                        )

                        await page.wait_for_timeout(
                            4000
                        )

                        print(
                            "Clicked:",
                            text
                        )

                        clicked = True

                        break

                except Exception:
                    pass

            if clicked:
                break


    # ========================================================
    # RESOURCE URL DARI PERFORMANCE API
    # ========================================================

    try:

        performance_urls = await page.evaluate(
            """
            () => performance
                .getEntriesByType('resource')
                .map(x => x.name)
            """
        )

        for url in performance_urls:

            resource_urls.add(
                clean_url(url)
            )

    except Exception as e:

        print(
            "Performance scan warning:",
            e
        )


    # ========================================================
    # SERVICE + ITEM IDS
    # ========================================================

    services = set()
    item_ids = set()

    for url in resource_urls:

        match = ARC_SERVICE_RE.search(
            url
        )

        if match:

            services.add(
                clean_url(
                    match.group(0)
                )
            )

        item_ids.update(
            ITEM_ID_RE.findall(
                url
            )
        )


    print(
        "Resources:",
        len(resource_urls)
    )

    print(
        "ArcGIS services:",
        len(services)
    )

    print(
        "ArcGIS item IDs:",
        len(item_ids)
    )


    return {
        "entry_url":
            entry_url,

        "final_url":
            clean_url(
                page.url
            ),

        "title":
            await page.title(),

        "resource_urls":
            sorted(
                resource_urls
            ),

        "service_urls":
            sorted(
                services
            ),

        "item_ids":
            sorted(
                item_ids
            )
    }

## 5. ArcGIS Portal item discovery

In [7]:
def portal_candidates_from_url(url):
    """
    Dari URL aplikasi ArcGIS Enterprise, tebak base portal:
    .../portal/apps/... -> .../portal
    .../gisportal/apps/... -> .../gisportal
    """
    clean = clean_url(url)

    for marker in [
        "/portal/",
        "/gisportal/"
    ]:

        if marker in clean:

            prefix = clean.split(
                marker
            )[0]

            segment = marker.strip(
                "/"
            )

            return (
                prefix +
                "/" +
                segment
            )

    return None


async def discover_arcgis_items_async(
    request_context,
    portal_base,
    seed_item_ids,
    max_rounds=4,
    max_items=150
):

    if not portal_base:

        return {
            "configs": [],
            "services": set(),
            "item_ids": set()
        }


    known_ids = set(
        seed_item_ids
    )

    processed = set()

    configs = []

    services = set()


    for round_idx in range(
        max_rounds
    ):

        pending = list(
            known_ids -
            processed
        )[:max_items]


        if not pending:
            break


        print(
            f"Portal discovery round "
            f"{round_idx + 1}: "
            f"{len(pending)} items"
        )


        for item_id in pending:

            processed.add(
                item_id
            )


            meta_url = (
                f"{portal_base}/"
                f"sharing/rest/content/items/"
                f"{item_id}"
            )


            data_url = (
                f"{portal_base}/"
                f"sharing/rest/content/items/"
                f"{item_id}/data"
            )


            meta = await request_json_async(
                request_context,
                meta_url,
                {
                    "f": "json"
                }
            )


            data = await request_json_async(
                request_context,
                data_url,
                {
                    "f": "json"
                }
            )


            configs.append({
                "item_id":
                    item_id,

                "metadata":
                    meta,

                "data":
                    data
            })


            services.update(
                extract_service_urls(
                    meta
                )
            )


            services.update(
                extract_service_urls(
                    data
                )
            )


            known_ids.update(
                extract_item_ids(
                    meta
                )
            )


            known_ids.update(
                extract_item_ids(
                    data
                )
            )


    return {
        "configs":
            configs,

        "services":
            services,

        "item_ids":
            known_ids
    }

## 6. Layer inspection & scoring

In [8]:
async def inspect_service_layers_async(
    request_context,
    service_url
):

    clean = clean_url(
        service_url
    )


    layer_id = explicit_layer_id(
        clean
    )


    # ========================================================
    # URL SUDAH LANGSUNG MENUJU LAYER
    # ========================================================

    if layer_id is not None:

        meta = await request_json_async(
            request_context,
            clean,
            {
                "f": "pjson"
            }
        )

        return [
            {
                "url":
                    clean,

                "metadata":
                    meta
            }
        ]


    # ========================================================
    # SERVICE ROOT
    # ========================================================

    root = service_root(
        clean
    )


    service_meta = await request_json_async(
        request_context,
        root,
        {
            "f": "pjson"
        }
    )


    # ========================================================
    # AUTH CHECK
    # ========================================================

    if auth_error(
        service_meta
    ):

        return [
            {
                "url":
                    root,

                "metadata":
                    service_meta
            }
        ]


    # ========================================================
    # INVALID METADATA
    # ========================================================

    if not isinstance(
        service_meta,
        dict
    ):

        return []


    # ========================================================
    # LAYERS
    #
    # beberapa service:
    #
    # "layers": null
    # ========================================================

    layers = (
        service_meta.get(
            "layers"
        )
        or []
    )


    output = []


    for layer in layers:

        if not isinstance(
            layer,
            dict
        ):
            continue


        layer_id = (
            layer.get(
                "id"
            )
        )


        if layer_id is None:
            continue


        layer_url = (
            f"{root}/{layer_id}"
        )


        try:

            meta = await request_json_async(
                request_context,
                layer_url,
                {
                    "f": "pjson"
                }
            )

        except Exception as e:

            meta = {
                "_metadata_error":
                    str(e)
            }


        output.append({
            "url":
                layer_url,

            "metadata":
                meta
        })


    return output


def score_layer(
    metadata,
    name_keywords,
    expected_fields
):

    # Metadata invalid
    if not isinstance(
        metadata,
        dict
    ):
        return -999


    # Layer butuh autentikasi
    if auth_error(
        metadata
    ):
        return -100


    # ========================================================
    # BASIC METADATA
    # ========================================================

    name = str(
        metadata.get(
            "name"
        )
        or ""
    )

    geometry_type = (
        metadata.get(
            "geometryType"
        )
        or ""
    )


    # ========================================================
    # FIELDS
    #
    # Beberapa ArcGIS service mengembalikan:
    #
    # "fields": null
    #
    # jadi HARUS pakai `or []`
    # ========================================================

    raw_fields = (
        metadata.get(
            "fields"
        )
        or []
    )


    fields = []

    for field in raw_fields:

        if not isinstance(
            field,
            dict
        ):
            continue

        field_name = (
            field.get(
                "name"
            )
            or ""
        )

        fields.append(
            str(
                field_name
            )
        )


    # ========================================================
    # SEARCHABLE STRING
    # ========================================================

    searchable = (
        name +
        " " +
        " ".join(
            fields
        )
    ).lower()


    score = 0


    # ========================================================
    # NAME / KEYWORD SCORE
    # ========================================================

    for keyword in name_keywords:

        if (
            keyword.lower()
            in searchable
        ):

            score += 8


    # ========================================================
    # EXPECTED FIELD SCORE
    # ========================================================

    lower_fields = {
        field.lower()
        for field
        in fields
    }


    for expected in expected_fields:

        if (
            expected.lower()
            in lower_fields
        ):

            score += 3


    # ========================================================
    # GEOMETRY SCORE
    # ========================================================

    if (
        geometry_type ==
        "esriGeometryPolygon"
    ):

        score += 10

    else:

        score -= 10


    # ========================================================
    # QUERY CAPABILITY
    # ========================================================

    capabilities = str(
        metadata.get(
            "capabilities"
        )
        or ""
    ).lower()


    if "query" in capabilities:

        score += 3


    return score


async def discover_best_layer_async(
    request_context,
    candidate_service_urls,
    name_keywords,
    expected_fields,
    min_score=15
):

    inspected = []


    urls = sorted(
        set(
            clean_url(
                url
            )
            for url
            in candidate_service_urls
            if url
        )
    )


    print(
        "Inspecting services:",
        len(urls)
    )


    for idx, service_url in enumerate(
        urls,
        start=1
    ):

        print(
            f"[{idx}/{len(urls)}]",
            service_url
        )


        try:

            layer_records = (
                await inspect_service_layers_async(
                    request_context,
                    service_url
                )
            )


        except Exception as e:

            inspected.append({

                "url":
                    service_url,

                "score":
                    -999,

                "name":
                    None,

                "geometry_type":
                    None,

                "fields":
                    [],

                "auth_required":
                    False,

                "error":
                    str(e),

                "metadata":
                    {}
            })

            continue


        for record in layer_records:

            metadata = (
                record.get(
                    "metadata"
                )
                or {}
            )


            # =================================================
            # SCORE
            # =================================================

            try:

                score = score_layer(
                    metadata,
                    name_keywords,
                    expected_fields
                )

            except Exception as e:

                print(
                    "  Skip malformed layer:",
                    record.get(
                        "url"
                    ),
                    "|",
                    e
                )

                score = -999


            # =================================================
            # SAFE FIELDS
            # =================================================

            raw_fields = (
                metadata.get(
                    "fields"
                )
                or []
            ) if isinstance(
                metadata,
                dict
            ) else []


            fields = []

            for field in raw_fields:

                if not isinstance(
                    field,
                    dict
                ):
                    continue

                name = (
                    field.get(
                        "name"
                    )
                    or ""
                )

                fields.append(
                    str(name)
                )


            # =================================================
            # RECORD
            # =================================================

            item = {

                "url":
                    record.get(
                        "url"
                    ),

                "score":
                    score,

                "name":
                    (
                        metadata.get(
                            "name"
                        )
                        if isinstance(
                            metadata,
                            dict
                        )
                        else None
                    ),

                "geometry_type":
                    (
                        metadata.get(
                            "geometryType"
                        )
                        if isinstance(
                            metadata,
                            dict
                        )
                        else None
                    ),

                "fields":
                    fields,

                "auth_required":
                    auth_error(
                        metadata
                    )
                    if isinstance(
                        metadata,
                        dict
                    )
                    else False,

                "metadata":
                    metadata
            }


            inspected.append(
                item
            )


    # ========================================================
    # SORT
    # ========================================================

    inspected.sort(
        key=lambda x:
        x.get(
            "score",
            -999
        ),
        reverse=True
    )


    # ========================================================
    # VALID CANDIDATES
    # ========================================================

    valid_candidates = [

        item

        for item
        in inspected

        if (
            item.get(
                "score",
                -999
            )
            >= min_score
        )
    ]


    selected = (
        valid_candidates[0]
        if valid_candidates
        else None
    )


    return (
        selected,
        inspected
    )

## 7. Query seluruh polygon Kalimantan

In [9]:
def build_province_where(
    fields,
    dataset_type
):
    lower_map = {
        str(f).lower():
            str(f)
        for f in fields
    }

    # --------------------------------------------------------
    # WIUP
    # Official dictionary commonly uses PROV / IDPROV / ISLAND.
    # --------------------------------------------------------

    if dataset_type == "wiup":

        if "prov" in lower_map:

            field = lower_map[
                "prov"
            ]

            values = ",".join(
                "'" + x.replace(
                    "'",
                    "''"
                ) + "'"
                for x in KALIMANTAN_PROV_NAMES
            )

            return (
                f"{field} IN ({values})"
            )

        if "island" in lower_map:

            field = lower_map[
                "island"
            ]

            return (
                f"UPPER({field}) "
                "LIKE '%KALIMANTAN%'"
            )

    # --------------------------------------------------------
    # PBPH
    # Official SIGAP dictionary commonly uses KODE_PROV.
    # --------------------------------------------------------

    if dataset_type == "pbph":

        if "kode_prov" in lower_map:

            field = lower_map[
                "kode_prov"
            ]

            # Type field bisa text/numeric.
            return (
                f"{field} IN "
                "('61','62','63','64','65',"
                "61,62,63,64,65)"
            )

    return "1=1"


async def query_layer_all_features_async(
    request_context,
    layer_url,
    layer_metadata,
    dataset_type
):

    fields = [
        field.get(
            "name",
            ""
        )
        for field
        in layer_metadata.get(
            "fields",
            []
        )
    ]


    where = build_province_where(
        fields,
        dataset_type
    )


    common = {

        "where":
            where,

        "geometry":
            json.dumps(
                KALIMANTAN_BBOX
            ),

        "geometryType":
            "esriGeometryEnvelope",

        "spatialRel":
            "esriSpatialRelIntersects",

        "inSR":
            4326
    }


    print(
        "WHERE:",
        where
    )


    # ========================================================
    # OBJECT IDS
    # ========================================================

    ids_result = await post_json_async(

        request_context,

        layer_url +
        "/query",

        {
            **common,

            "returnIdsOnly":
                "true",

            "f":
                "json"
        }
    )


    if auth_error(
        ids_result
    ):

        raise PermissionError(
            "Layer resmi ditemukan, tetapi "
            "server meminta autentikasi/token."
        )


    if arcgis_error(
        ids_result
    ):

        print(
            "WHERE provinsi gagal, "
            "retry bbox-only..."
        )


        ids_result = await post_json_async(

            request_context,

            layer_url +
            "/query",

            {
                **common,

                "where":
                    "1=1",

                "returnIdsOnly":
                    "true",

                "f":
                    "json"
            }
        )


    if auth_error(
        ids_result
    ):

        raise PermissionError(
            "Layer resmi membutuhkan "
            "autentikasi."
        )


    if arcgis_error(
        ids_result
    ):

        raise RuntimeError(
            arcgis_error(
                ids_result
            )
        )


    object_ids = (
        ids_result.get(
            "objectIds"
        )
        or []
    )


    print(
        "Object IDs:",
        len(object_ids)
    )


    if not object_ids:

        raise RuntimeError(
            "Tidak ada polygon Kalimantan "
            "yang dikembalikan layer."
        )


    max_record_count = int(
        layer_metadata.get(
            "maxRecordCount",
            500
        )
        or 500
    )


    chunk_size = min(
        max_record_count,
        500
    )


    all_features = []


    # ========================================================
    # DOWNLOAD PER CHUNK
    # ========================================================

    for start in range(
        0,
        len(object_ids),
        chunk_size
    ):

        chunk = object_ids[
            start:
            start + chunk_size
        ]


        print(
            f"Download "
            f"{start + 1}-"
            f"{min(start + chunk_size, len(object_ids))}"
            f"/{len(object_ids)}"
        )


        result = await post_json_async(

            request_context,

            layer_url +
            "/query",

            {
                "objectIds":
                    ",".join(
                        map(
                            str,
                            chunk
                        )
                    ),

                "outFields":
                    "*",

                "returnGeometry":
                    "true",

                "outSR":
                    4326,

                "f":
                    "geojson"
            }
        )


        if auth_error(
            result
        ):

            raise PermissionError(
                "Token diperlukan saat "
                "mengambil geometry."
            )


        if arcgis_error(
            result
        ):

            raise RuntimeError(
                arcgis_error(
                    result
                )
            )


        all_features.extend(
            result.get(
                "features",
                []
            )
        )


    return all_features

# PART A — WIUP ESDM / MOMI

Kamus Data Spasial ESDM mendefinisikan **WIUP sebagai polygon**, diperbarui bulanan, dengan atribut penting seperti:
- `PROV`
- `KAB`
- `ISLAND`
- `NMOPRT` — nama perusahaan
- `TIPOPR` — jenis badan usaha
- `SKBLOK` — nomor SK
- `TIPTMB` — jenis izin
- `DATSTR`
- `DATEND`

Notebook mencari field tersebut pada service yang benar-benar dipakai public map resmi.

## 8. Buka public MOMI/ESDM map dan discover WIUP

In [10]:
WIUP_NAME_KEYWORDS = [
    "wiup",
    "wilayah izin usaha pertambangan",
    "izin usaha pertambangan"
]

WIUP_EXPECTED_FIELDS = [
    "ISSUER",
    "PROV",
    "IDPROV",
    "KAB",
    "IDKAB",
    "ISLAND",
    "NMOPRT",
    "TIPOPR",
    "ADOPRT",
    "SKBLOK",
    "TIPTMB",
    "DATSTR",
    "DATEND"
]


async def run_wiup():

    async with async_playwright() as p:

        # ====================================================
        # BROWSER
        # ====================================================

        browser = await p.chromium.launch(executable_path='C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe', 
            headless=True
        )

        context = await browser.new_context(
            ignore_https_errors=True,
            viewport={
                "width": 1600,
                "height": 1000
            }
        )

        page = await context.new_page()


        # ====================================================
        # OPEN MOMI / ESDM PUBLIC APP
        # ====================================================

        wiup_traffic = await collect_app_traffic_async(
            page=page,
            entry_url=WIUP_ENTRY_URL,
            wait_ms=30_000,
            interaction_texts=[
                "WIUP",
                "Pertambangan",
                "Minerba",
                "Layer"
            ]
        )


        # ====================================================
        # PORTAL DISCOVERY
        # ====================================================

        portal_base = portal_candidates_from_url(
            wiup_traffic[
                "final_url"
            ]
        )

        print(
            "Portal base:",
            portal_base
        )


        if portal_base:

            portal_result = await discover_arcgis_items_async(
                context.request,
                portal_base,
                wiup_traffic[
                    "item_ids"
                ]
            )

        else:

            portal_result = {
                "configs": [],
                "services": set(),
                "item_ids": set()
            }


        # ====================================================
        # COLLECT SERVICE URL
        # ====================================================

        service_urls = set(
            wiup_traffic[
                "service_urls"
            ]
        )

        service_urls.update(
            portal_result[
                "services"
            ]
        )


        print(
            "\nTotal WIUP candidate services:",
            len(service_urls)
        )


        # ====================================================
        # DISCOVER BEST WIUP LAYER
        # ====================================================

        selected_wiup, inspected_wiup = (
            await discover_best_layer_async(
                context.request,
                service_urls,
                WIUP_NAME_KEYWORDS,
                WIUP_EXPECTED_FIELDS,
                min_score=20
            )
        )


        # ====================================================
        # PRINT TOP CANDIDATES
        # ====================================================

        print(
            "\nTOP WIUP CANDIDATES"
        )

        for candidate in inspected_wiup[:10]:

            print(
                candidate.get(
                    "score"
                ),
                "|",
                candidate.get(
                    "name"
                ),
                "|",
                candidate.get(
                    "geometry_type"
                ),
                "|",
                candidate.get(
                    "url"
                )
            )


        # ====================================================
        # SAVE DISCOVERY REPORT
        # ====================================================

        safe_report = {

            "entry_url":
                wiup_traffic[
                    "entry_url"
                ],

            "final_url":
                wiup_traffic[
                    "final_url"
                ],

            "portal_base":
                portal_base,

            "service_urls":
                sorted(
                    clean_url(x)
                    for x
                    in service_urls
                ),

            "top_layers": [

                {
                    "url":
                        x.get(
                            "url"
                        ),

                    "score":
                        x.get(
                            "score"
                        ),

                    "name":
                        x.get(
                            "name"
                        ),

                    "geometry_type":
                        x.get(
                            "geometry_type"
                        ),

                    "fields":
                        x.get(
                            "fields",
                            []
                        ),

                    "auth_required":
                        x.get(
                            "auth_required",
                            False
                        )
                }

                for x
                in inspected_wiup[:30]
            ]
        }


        WIUP_DISCOVERY_OUT.write_text(
            json.dumps(
                safe_report,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )


        print(
            "\nDiscovery report:",
            WIUP_DISCOVERY_OUT
        )


        # ====================================================
        # CHECK SELECTED LAYER
        # ====================================================

        if selected_wiup is None:

            await browser.close()

            raise RuntimeError(
                "Tidak menemukan layer WIUP polygon resmi "
                "yang cukup meyakinkan dari public app. "
                "Periksa wiup_discovery.json."
            )


        print(
            "\n=============================="
        )

        print(
            "SELECTED WIUP"
        )

        print(
            "Name :",
            selected_wiup[
                "name"
            ]
        )

        print(
            "URL  :",
            selected_wiup[
                "url"
            ]
        )

        print(
            "Score:",
            selected_wiup[
                "score"
            ]
        )

        print(
            "Geometry:",
            selected_wiup[
                "geometry_type"
            ]
        )

        print(
            "=============================="
        )


        # ====================================================
        # DOWNLOAD WIUP FEATURES
        # ====================================================

        wiup_features = await query_layer_all_features_async(
            context.request,
            selected_wiup[
                "url"
            ],
            selected_wiup[
                "metadata"
            ],
            dataset_type="wiup"
        )


        # ====================================================
        # SAVE GEOJSON
        # ====================================================

        save_feature_collection(
            wiup_features,
            WIUP_OUT,
            "WIUP Kalimantan - Official ESDM"
        )


        print(
            "\nJumlah WIUP features:",
            len(
                wiup_features
            )
        )


        await browser.close()


        return {
            "selected_layer":
                selected_wiup[
                    "name"
                ],

            "selected_url":
                selected_wiup[
                    "url"
                ],

            "features":
                len(
                    wiup_features
                )
        }

In [11]:
wiup_result = await run_wiup()

wiup_result

NotImplementedError: 

## 9. Validasi WIUP

In [12]:
# ============================================================
# VALIDASI HASIL WIUP
# ============================================================

wiup = gpd.read_file(
    WIUP_OUT
)

print(
    "Jumlah polygon:",
    len(wiup)
)

print(
    "CRS:",
    wiup.crs
)

print(
    "\nGeometry:"
)

display(
    wiup.geom_type
    .value_counts()
    .rename_axis(
        "geometry_type"
    )
    .reset_index(
        name="count"
    )
)


print(
    "\nColumns:"
)

print(
    wiup.columns.tolist()
)


# ============================================================
# CEK ATRIBUT RESMI YANG TERSEDIA
# ============================================================

wiup_expected_available = [
    col
    for col
    in WIUP_EXPECTED_FIELDS
    if col in wiup.columns
]

print(
    "\nExpected official attributes available:"
)

print(
    wiup_expected_available
)


# ============================================================
# TAMPILKAN ATRIBUT PENTING
# ============================================================

important_cols = [
    col
    for col in [
        "NMOPRT",   # nama perusahaan
        "TIPOPR",   # jenis badan usaha
        "PROV",     # provinsi
        "KAB",      # kabupaten
        "SKBLOK",   # nomor SK
        "TIPTMB",   # jenis izin
        "DATSTR",   # tanggal mulai
        "DATEND",   # tanggal akhir
        "ISSUER",
        "ISLAND",
        "geometry"
    ]
    if col in wiup.columns
]


if important_cols:

    display(
        wiup[
            important_cols
        ].head(20)
    )

else:

    display(
        wiup.head(20)
    )

# ============================================================
# CEK PROVINSI
# ============================================================

if "PROV" in wiup.columns:

    print(
        "\nProvinsi yang ditemukan:"
    )

    display(
        wiup["PROV"]
        .value_counts(
            dropna=False
        )
        .reset_index()
        .rename(
            columns={
                "PROV": "province",
                "count": "records"
            }
        )
    )

# ============================================================
# CEK NAMA PERUSAHAAN
# ============================================================

if "NMOPRT" in wiup.columns:

    print(
        "\nTop perusahaan / operator:"
    )

    display(
        wiup["NMOPRT"]
        .value_counts(
            dropna=False
        )
        .head(20)
        .reset_index()
        .rename(
            columns={
                "NMOPRT": "company",
                "count": "records"
            }
        )
    )

DataSourceError: \\wsl.localhost\Ubuntu\home\adit\projects\compfest_academy_2\data\official_concessions\wiup_kalimantan_official.geojson: No such file or directory

In [13]:
print("Provinsi yang ada:")

display(
    wiup["nama_prov"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(
        columns={
            "nama_prov": "provinsi",
            "count": "jumlah_wiup"
        }
    )
)

Provinsi yang ada:


NameError: name 'wiup' is not defined

# PART B — PBPH Kementerian Kehutanan / SIGAP

Untuk PBPH, notebook **tidak memakai domain geoportal KLHK lama** yang sudah gagal DNS.

Strateginya:
1. buka `sigap.kehutanan.go.id`,
2. cari link **Peta Interaktif** resmi yang aktif saat notebook dijalankan,
3. buka link tersebut dengan Playwright,
4. discover layer PBPH berdasarkan atribut resmi seperti:
   - `NAMOBJ`
   - `PBPH_ID`
   - `KODE_PROV`
   - `NO_SK`
   - `TGL_SK`
   - `LSSK`
5. download polygon jika akses publik mengizinkan.

## 10. Cari link Peta Interaktif SIGAP yang aktif

In [14]:
async def find_sigap_map_url_async(
    page,
    entry_url
):
    print(
        "Opening SIGAP:",
        entry_url
    )

    await page.goto(
        entry_url,
        wait_until="domcontentloaded",
        timeout=180_000
    )

    await page.wait_for_timeout(
        10_000
    )

    hrefs = await page.locator(
        "a[href]"
    ).evaluate_all(
        """
        els => els.map(
            a => ({
                text: (a.innerText || '').trim(),
                href: a.href
            })
        )
        """
    )

    scored = []

    for item in hrefs:

        text = (
            item.get(
                "text",
                ""
            )
            .lower()
        )

        href = item.get(
            "href",
            ""
        )

        low_href = (
            href.lower()
        )

        score = 0

        if "peta interaktif" in text:
            score += 20

        if "geoportal" in text:
            score += 10

        if "geoportal" in low_href:
            score += 10

        if (
            "planologi.kehutanan.go.id"
            in low_href
        ):
            score += 20

        if (
            "/apps/"
            in low_href
            or
            "experience" in low_href
        ):
            score += 5

        if score > 0:

            scored.append(
                (
                    score,
                    href,
                    item.get(
                        "text",
                        ""
                    )
                )
            )

    scored.sort(
        reverse=True
    )

    print(
        "\nSIGAP map candidates:"
    )

    for candidate in scored[:10]:

        print(
            candidate
        )

    if not scored:

        raise RuntimeError(
            "Tidak menemukan link Peta Interaktif "
            "dari halaman SIGAP."
        )

    print(
        "\nSelected SIGAP map:"
    )

    print(
        scored[0]
    )

    return scored[0][1]

## 11. Discover & download PBPH resmi

In [15]:
PBPH_NAME_KEYWORDS = [
    "pbph",
    "perizinan berusaha pemanfaatan hutan",
    "pemanfaatan hutan",
    "rkuph"
]

PBPH_EXPECTED_FIELDS = [
    "NAMOBJ",
    "PBPH_ID",
    "KODE_PROV",
    "NO_SK",
    "TGL_SK",
    "LSSK"
]


async def run_pbph():

    async with async_playwright() as p:

        # ====================================================
        # BROWSER
        # ====================================================

        browser = await p.chromium.launch(executable_path='C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe', 
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )

        context = await browser.new_context(
            ignore_https_errors=True,
            viewport={
                "width": 1600,
                "height": 1000
            }
        )


        # ====================================================
        # BUKA SIGAP
        # ====================================================

        sigap_page = await context.new_page()

        SIGAP_MAP_URL = await find_sigap_map_url_async(
            sigap_page,
            SIGAP_ENTRY_URL
        )

        print(
            "\nUsing official map URL:",
            SIGAP_MAP_URL
        )


        # ====================================================
        # BUKA PETA INTERAKTIF
        # ====================================================

        page = await context.new_page()

        pbph_traffic = await collect_app_traffic_async(
            page=page,
            entry_url=SIGAP_MAP_URL,
            wait_ms=30_000,
            interaction_texts=[
                "PBPH",
                "Pemanfaatan Hutan",
                "Perizinan",
                "Layer"
            ]
        )


        # ====================================================
        # DISCOVER PORTAL BASE
        # ====================================================

        portal_base = portal_candidates_from_url(
            pbph_traffic[
                "final_url"
            ]
        )

        print(
            "Portal base:",
            portal_base
        )


        # ====================================================
        # DISCOVER ARC GIS ITEMS
        # ====================================================

        if portal_base:

            portal_result = await discover_arcgis_items_async(
                context.request,
                portal_base,
                pbph_traffic[
                    "item_ids"
                ]
            )

        else:

            portal_result = {
                "configs": [],
                "services": set(),
                "item_ids": set()
            }


        # ====================================================
        # GABUNGKAN SEMUA SERVICE URL
        # ====================================================

        service_urls = set(
            pbph_traffic[
                "service_urls"
            ]
        )

        service_urls.update(
            portal_result[
                "services"
            ]
        )


        print(
            "\nTotal PBPH candidate services:",
            len(service_urls)
        )


        # ====================================================
        # CARI LAYER PBPH TERBAIK
        # ====================================================

        selected_pbph, inspected_pbph = (
            await discover_best_layer_async(
                context.request,
                service_urls,
                PBPH_NAME_KEYWORDS,
                PBPH_EXPECTED_FIELDS,
                min_score=20
            )
        )


        # ====================================================
        # PRINT TOP CANDIDATES
        # ====================================================

        print(
            "\nTOP PBPH CANDIDATES"
        )

        for candidate in inspected_pbph[:10]:

            print(
                candidate.get(
                    "score"
                ),
                "|",
                candidate.get(
                    "name"
                ),
                "|",
                candidate.get(
                    "geometry_type"
                ),
                "|",
                candidate.get(
                    "url"
                )
            )


        # ====================================================
        # SAVE DISCOVERY REPORT
        # ====================================================

        safe_report = {

            "sigap_entry_url":
                SIGAP_ENTRY_URL,

            "map_url":
                clean_url(
                    SIGAP_MAP_URL
                ),

            "final_url":
                pbph_traffic[
                    "final_url"
                ],

            "portal_base":
                portal_base,

            "service_urls":
                sorted(
                    clean_url(x)
                    for x
                    in service_urls
                ),

            "top_layers": [

                {
                    "url":
                        x.get(
                            "url"
                        ),

                    "score":
                        x.get(
                            "score"
                        ),

                    "name":
                        x.get(
                            "name"
                        ),

                    "geometry_type":
                        x.get(
                            "geometry_type"
                        ),

                    "fields":
                        x.get(
                            "fields",
                            []
                        ),

                    "auth_required":
                        x.get(
                            "auth_required",
                            False
                        )
                }

                for x
                in inspected_pbph[:30]
            ]
        }


        PBPH_DISCOVERY_OUT.write_text(
            json.dumps(
                safe_report,
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )


        print(
            "\nDiscovery report:",
            PBPH_DISCOVERY_OUT
        )


        # ====================================================
        # CHECK SELECTED
        # ====================================================

        if selected_pbph is None:

            await browser.close()

            raise RuntimeError(
                "Tidak menemukan layer PBPH polygon resmi "
                "yang cukup meyakinkan dari Peta Interaktif SIGAP. "
                "Periksa pbph_discovery.json."
            )


        print(
            "\n=============================="
        )

        print(
            "SELECTED PBPH"
        )

        print(
            "Name :",
            selected_pbph[
                "name"
            ]
        )

        print(
            "URL  :",
            selected_pbph[
                "url"
            ]
        )

        print(
            "Score:",
            selected_pbph[
                "score"
            ]
        )

        print(
            "Geometry:",
            selected_pbph[
                "geometry_type"
            ]
        )

        print(
            "=============================="
        )


        # ====================================================
        # DOWNLOAD PBPH
        # ====================================================

        pbph_features = await query_layer_all_features_async(
            context.request,
            selected_pbph[
                "url"
            ],
            selected_pbph[
                "metadata"
            ],
            dataset_type="pbph"
        )


        # ====================================================
        # SAVE GEOJSON
        # ====================================================

        save_feature_collection(
            pbph_features,
            PBPH_OUT,
            "PBPH Kalimantan - Official Kementerian Kehutanan"
        )


        print(
            "\nJumlah PBPH features:",
            len(
                pbph_features
            )
        )


        await browser.close()


        return {
            "selected_layer":
                selected_pbph[
                    "name"
                ],

            "selected_url":
                selected_pbph[
                    "url"
                ],

            "features":
                len(
                    pbph_features
                )
        }

In [16]:
pbph_result = await run_pbph()

pbph_result

NotImplementedError: 

## 12. Validasi PBPH

In [17]:
pbph = gpd.read_file(
    PBPH_OUT
)

print(
    "Jumlah polygon:",
    len(pbph)
)

print(
    "CRS:",
    pbph.crs
)

display(
    pbph.geom_type
    .value_counts()
    .rename_axis(
        "geometry_type"
    )
    .reset_index(
        name="count"
    )
)

print(
    "\nColumns:"
)

print(
    pbph.columns.tolist()
)

DataSourceError: \\wsl.localhost\Ubuntu\home\adit\projects\compfest_academy_2\data\official_concessions\pbph_kalimantan_official.geojson: No such file or directory

## 13. QA gabungan

In [18]:
qa = []

for dataset_name, path, expected_fields in [
    (
        "WIUP",
        WIUP_OUT,
        WIUP_EXPECTED_FIELDS
    ),
    (
        "PBPH",
        PBPH_OUT,
        PBPH_EXPECTED_FIELDS
    )
]:

    if not path.exists():

        qa.append({
            "dataset":
                dataset_name,

            "exists":
                False,

            "features":
                None,

            "geometry":
                None,

            "expected_fields_found":
                None
        })

        continue

    gdf = gpd.read_file(
        path
    )

    found_fields = [
        field
        for field in expected_fields
        if field in gdf.columns
    ]

    qa.append({
        "dataset":
            dataset_name,

        "exists":
            True,

        "features":
            len(gdf),

        "geometry":
            ", ".join(
                sorted(
                    gdf.geom_type
                    .dropna()
                    .unique()
                )
            ),

        "expected_fields_found":
            ", ".join(
                found_fields
            )
    })


qa_df = pd.DataFrame(
    qa
)

display(
    qa_df
)

,dataset,exists,features,geometry,expected_fields_found
0,WIUP,False,None,None,None
1,PBPH,False,None,None,None


## 14. Manifest sumber resmi

In [19]:
from datetime import datetime, timezone

manifest = {
    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "method":
        "Playwright public-browser discovery + official ArcGIS service query",

    "policy":
        (
            "No proxy dataset fallback. "
            "No authentication bypass. "
            "If official public access is restricted, acquisition stops."
        ),

    "wiup": {
        "entry_point":
            WIUP_ENTRY_URL,

        "output":
            str(
                WIUP_OUT
            ),

        "discovery_report":
            str(
                WIUP_DISCOVERY_OUT
            ),

        "producer":
            "Direktorat Jenderal Mineral dan Batubara, Kementerian ESDM"
    },

    "pbph": {
        "entry_point":
            SIGAP_ENTRY_URL,

        "output":
            str(
                PBPH_OUT
            ),

        "discovery_report":
            str(
                PBPH_DISCOVERY_OUT
            ),

        "producer":
            "Kementerian Kehutanan / SIGAP"
    },

    "legal_note":
        (
            "A hotspot spatially intersecting an official permit polygon "
            "does not establish fire causation or legal liability."
        )
}

manifest_path = (
    OUTPUT_DIR /
    "acquisition_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    manifest_path
)

Saved: \\wsl.localhost\Ubuntu\home\adit\projects\compfest_academy_2\data\official_concessions\acquisition_manifest.json


# responders

In [20]:
from pathlib import Path
from urllib.parse import (
    quote_plus,
    urlparse,
    parse_qs,
    unquote
)

import asyncio
import json
import re

import pandas as pd
import geopandas as gpd

from shapely.geometry import Point
from playwright.async_api import async_playwright


# ============================================================
# OUTPUT
# ============================================================

RESPONDER_DIR = (
    PROJECT_ROOT /
    "responders"
)

RESPONDER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RESPONDER_AUDIT_OUT = (
    RESPONDER_DIR /
    "responders_playwright_audit.csv"
)

RESPONDER_GEOJSON_OUT = (
    RESPONDER_DIR /
    "responders_official_playwright.geojson"
)


# ============================================================
# PLAYWRIGHT CONFIG
# ============================================================

HEADLESS = True

SEARCH_WAIT_MS = 1500
PAGE_WAIT_MS = 1800

MAX_OFFICIAL_PAGES_PER_TARGET = 4

RETRY_NOT_FOUND = True


# ============================================================
# KALIMANTAN BBOX
# ============================================================

KAL_BBOX = {
    "west": 108.0,
    "south": -4.9,
    "east": 119.9,
    "north": 4.9
}


def valid_kalimantan_coordinate(
    lat,
    lon
):
    try:
        lat = float(lat)
        lon = float(lon)

    except Exception:
        return None

    if not (
        KAL_BBOX["south"]
        <= lat
        <= KAL_BBOX["north"]
    ):
        return None

    if not (
        KAL_BBOX["west"]
        <= lon
        <= KAL_BBOX["east"]
    ):
        return None

    return (
        lat,
        lon
    )

In [21]:
ADMIN2 = {

    "Kalimantan Barat": [
        "Kabupaten Bengkayang",
        "Kabupaten Kapuas Hulu",
        "Kabupaten Kayong Utara",
        "Kabupaten Ketapang",
        "Kabupaten Kubu Raya",
        "Kabupaten Landak",
        "Kabupaten Melawi",
        "Kabupaten Mempawah",
        "Kabupaten Sambas",
        "Kabupaten Sanggau",
        "Kabupaten Sekadau",
        "Kabupaten Sintang",
        "Kota Pontianak",
        "Kota Singkawang"
    ],

    "Kalimantan Tengah": [
        "Kabupaten Barito Selatan",
        "Kabupaten Barito Timur",
        "Kabupaten Barito Utara",
        "Kabupaten Gunung Mas",
        "Kabupaten Kapuas",
        "Kabupaten Katingan",
        "Kabupaten Kotawaringin Barat",
        "Kabupaten Kotawaringin Timur",
        "Kabupaten Lamandau",
        "Kabupaten Murung Raya",
        "Kabupaten Pulang Pisau",
        "Kabupaten Seruyan",
        "Kabupaten Sukamara",
        "Kota Palangka Raya"
    ],

    "Kalimantan Selatan": [
        "Kabupaten Balangan",
        "Kabupaten Banjar",
        "Kabupaten Barito Kuala",
        "Kabupaten Hulu Sungai Selatan",
        "Kabupaten Hulu Sungai Tengah",
        "Kabupaten Hulu Sungai Utara",
        "Kabupaten Kotabaru",
        "Kabupaten Tabalong",
        "Kabupaten Tanah Bumbu",
        "Kabupaten Tanah Laut",
        "Kabupaten Tapin",
        "Kota Banjarbaru",
        "Kota Banjarmasin"
    ],

    "Kalimantan Timur": [
        "Kabupaten Berau",
        "Kabupaten Kutai Barat",
        "Kabupaten Kutai Kartanegara",
        "Kabupaten Kutai Timur",
        "Kabupaten Mahakam Ulu",
        "Kabupaten Paser",
        "Kabupaten Penajam Paser Utara",
        "Kota Balikpapan",
        "Kota Bontang",
        "Kota Samarinda"
    ],

    "Kalimantan Utara": [
        "Kabupaten Bulungan",
        "Kabupaten Malinau",
        "Kabupaten Nunukan",
        "Kabupaten Tana Tidung",
        "Kota Tarakan"
    ]
}

In [22]:
MANGGALA_TARGETS = [

    (
        "Kalimantan I",
        "Palangka Raya",
        "Kalimantan Tengah"
    ),

    (
        "Kalimantan II",
        "Kapuas",
        "Kalimantan Tengah"
    ),

    (
        "Kalimantan III",
        "Pangkalan Bun",
        "Kalimantan Tengah"
    ),

    (
        "Kalimantan IV",
        "Muara Teweh",
        "Kalimantan Tengah"
    ),

    (
        "Kalimantan V",
        "Banjar",
        "Kalimantan Selatan"
    ),

    (
        "Kalimantan VI",
        "Tanah Laut",
        "Kalimantan Selatan"
    ),

    (
        "Kalimantan VII",
        "Tanah Bumbu",
        "Kalimantan Selatan"
    ),

    (
        "Kalimantan VIII",
        "Pontianak",
        "Kalimantan Barat"
    ),

    (
        "Kalimantan IX",
        "Singkawang",
        "Kalimantan Barat"
    ),

    (
        "Kalimantan X",
        "Ketapang",
        "Kalimantan Barat"
    ),

    (
        "Kalimantan XI",
        "Sintang",
        "Kalimantan Barat"
    ),

    (
        "Kalimantan XII",
        "Paser",
        "Kalimantan Timur"
    ),

    (
        "Kalimantan XIII",
        "Sangkima",
        "Kalimantan Timur"
    )
]

In [23]:
TARGETS = []


# ============================================================
# MANGGALA AGNI
# ============================================================

for code, location, province in MANGGALA_TARGETS:

    TARGETS.append({

        "target_id":
            f"MA-{code.replace(' ', '-')}",

        "responder_type":
            "manggala_agni",

        "name":
            f"Daops Manggala Agni {code} / {location}",

        "area":
            location,

        "province":
            province,

        "authority":
            "Kementerian Kehutanan"
    })


# ============================================================
# BPBD PROVINSI
# ============================================================

for province in ADMIN2.keys():

    TARGETS.append({

        "target_id":
            "BPBD-PROV-" +
            province.upper()
            .replace(" ", "-"),

        "responder_type":
            "bpbd_province",

        "name":
            f"BPBD Provinsi {province}",

        "area":
            province,

        "province":
            province,

        "authority":
            "Pemerintah Provinsi"
    })


# ============================================================
# BPBD + DAMKAR KAB/KOTA
# ============================================================

for province, areas in ADMIN2.items():

    for area in areas:

        clean_id = (
            area.upper()
            .replace(" ", "-")
        )


        TARGETS.append({

            "target_id":
                f"BPBD-{clean_id}",

            "responder_type":
                "bpbd",

            "name":
                f"BPBD {area}",

            "area":
                area,

            "province":
                province,

            "authority":
                "Pemerintah Daerah"
        })


        TARGETS.append({

            "target_id":
                f"DAMKAR-{clean_id}",

            "responder_type":
                "damkar",

            "name":
                f"Pemadam Kebakaran {area}",

            "area":
                area,

            "province":
                province,

            "authority":
                "Pemerintah Daerah"
        })


targets_df = pd.DataFrame(
    TARGETS
)


print(
    "Total responder targets:",
    len(
        targets_df
    )
)


display(
    targets_df[
        "responder_type"
    ]
    .value_counts()
)

Total responder targets: 130


responder_type
damkar           56
bpbd             56
manggala_agni    13
bpbd_province     5
Name: count, dtype: int64

In [24]:
def is_official_government_url(url):

    if not url:
        return False

    try:

        host = (
            urlparse(url)
            .netloc
            .lower()
            .split(":")[0]
        )

    except Exception:

        return False


    # Hilangkan www.
    if host.startswith("www."):
        host = host[4:]


    # Pemerintah Indonesia
    if host.endswith(".go.id"):
        return True


    # Beberapa domain pusat
    allowed_domains = [
        "kehutanan.go.id",
        "menlhk.go.id"
    ]


    for domain in allowed_domains:

        if (
            host == domain
            or
            host.endswith("." + domain)
        ):
            return True


    return False

In [25]:
def decode_search_url(
    href
):

    if not href:
        return None


    href = unquote(
        href
    )


    # DuckDuckGo redirect
    if (
        "duckduckgo.com/l/"
        in href
    ):

        parsed = urlparse(
            href
        )

        query = parse_qs(
            parsed.query
        )

        uddg = query.get(
            "uddg"
        )

        if uddg:

            return unquote(
                uddg[0]
            )


    return href

In [26]:
from urllib.parse import (
    urlparse,
    parse_qs,
    unquote,
    quote_plus
)

import base64
import html


def decode_bing_result_url(href):

    if not href:
        return None


    href = html.unescape(
        href
    )

    href = unquote(
        href
    ).strip()


    if (
        href.startswith("javascript:")
        or
        href.startswith("mailto:")
        or
        href.startswith("#")
    ):
        return None


    try:

        parsed = urlparse(
            href
        )

    except Exception:

        return None


    host = (
        parsed.netloc
        .lower()
    )


    # ========================================================
    # KALAU BUKAN LINK BING -> LANGSUNG RETURN
    # ========================================================

    if (
        "bing.com"
        not in host
    ):

        return href


    # ========================================================
    # BING REDIRECT
    #
    # biasanya:
    # /ck/a?...&u=a1BASE64...
    # ========================================================

    params = parse_qs(
        parsed.query
    )


    candidate_values = []


    for key in [
        "u",
        "url",
        "r",
        "redir"
    ]:

        values = params.get(
            key,
            []
        )

        candidate_values.extend(
            values
        )


    for value in candidate_values:

        if not value:
            continue


        value = unquote(
            value
        )


        # ====================================================
        # SUDAH DIRECT URL
        # ====================================================

        if value.startswith(
            "http"
        ):

            return value


        # ====================================================
        # FORMAT BING:
        #
        # a1 + base64(url)
        # ====================================================

        encoded = value


        if encoded.startswith(
            "a1"
        ):

            encoded = (
                encoded[2:]
            )


        try:

            padding = (
                "="
                *
                (
                    -len(encoded)
                    % 4
                )
            )


            decoded = (
                base64.urlsafe_b64decode(
                    encoded
                    +
                    padding
                )
                .decode(
                    "utf-8",
                    errors="ignore"
                )
            )


            if decoded.startswith(
                "http"
            ):

                return decoded


        except Exception:

            pass


    # Tidak berhasil decode internal Bing
    return None

In [27]:
def build_search_queries(target):

    name = target["name"]
    area = target["area"]
    province = target["province"]
    responder_type = target["responder_type"]


    # Bersihkan prefix administrasi
    clean_area = (
        area
        .replace("Kabupaten ", "")
        .replace("Kota ", "")
        .strip()
    )


    if responder_type == "manggala_agni":

        return [

            f'Manggala Agni "{clean_area}" "{province}"',

            f'Daops Manggala Agni "{clean_area}"',

            f'Manggala Agni "{clean_area}" site:kehutanan.go.id',

            f'Manggala Agni "{clean_area}" site:menlhk.go.id',

            f'Manggala Agni "{clean_area}" site:go.id'
        ]


    if responder_type == "bpbd_province":

        return [

            f'BPBD "{province}" alamat',

            f'Badan Penanggulangan Bencana Daerah "{province}"',

            f'BPBD "{province}" site:go.id'
        ]


    if responder_type == "bpbd":

        return [

            f'BPBD "{clean_area}" "{province}"',

            f'Badan Penanggulangan Bencana Daerah "{clean_area}"',

            f'BPBD "{clean_area}" site:go.id'
        ]


    if responder_type == "damkar":

        return [

            f'Damkar "{clean_area}" "{province}"',

            f'Pemadam Kebakaran "{clean_area}" "{province}"',

            f'Dinas Pemadam Kebakaran "{clean_area}"',

            f'Damkar "{clean_area}" site:go.id'
        ]


    return [
        f'"{name}" site:go.id'
    ]

In [28]:
async def search_official_pages_playwright(
    page,
    target
):

    queries = (
        build_search_queries(
            target
        )
    )


    collected = []


    for query in queries:

        print(
            "   Bing:",
            query
        )


        search_url = (
            "https://www.bing.com/search?q="
            +
            quote_plus(
                query
            )
            +
            "&setlang=id-ID"
            +
            "&cc=id"
            +
            "&count=20"
        )


        # ====================================================
        # OPEN SEARCH
        # ====================================================

        success = False


        for attempt in range(
            2
        ):

            try:

                await page.goto(
                    search_url,
                    wait_until="domcontentloaded",
                    timeout=90_000
                )


                await page.wait_for_timeout(
                    2500
                )


                title = (
                    await page.title()
                )


                print(
                    "      title:",
                    title
                )


                success = True

                break


            except Exception as e:

                print(
                    "      attempt",
                    attempt + 1,
                    "failed:",
                    e
                )


                await page.wait_for_timeout(
                    2000
                )


        if not success:

            continue


        # ====================================================
        # RESULT LINKS
        # ====================================================

        selectors = [

            "li.b_algo h2 a[href]",

            "#b_results li.b_algo h2 a[href]"
        ]


        result_links = []


        for selector in selectors:

            try:

                links = (
                    await page.locator(
                        selector
                    )
                    .evaluate_all(
                        """
                        els => els.map(a => ({
                            href:
                                a.href || '',

                            text:
                                (a.innerText || '').trim()
                        }))
                        """
                    )
                )


            except Exception:

                links = []


            if links:

                result_links.extend(
                    links
                )


        # ====================================================
        # FALLBACK
        #
        # kalau Bing DOM berubah
        # ====================================================

        if not result_links:

            try:

                result_links = (
                    await page.locator(
                        "#b_results a[href]"
                    )
                    .evaluate_all(
                        """
                        els => els.map(a => ({
                            href:
                                a.href || '',

                            text:
                                (a.innerText || '').trim()
                        }))
                        """
                    )
                )


            except Exception:

                result_links = []


        print(
            "      raw result links:",
            len(
                result_links
            )
        )


        # ====================================================
        # DECODE BING REDIRECT
        # ====================================================

        decoded_count = 0


        for item in result_links:

            raw_href = (
                item.get(
                    "href"
                )
            )


            href = (
                decode_bing_result_url(
                    raw_href
                )
            )


            if not href:

                continue


            decoded_count += 1


            # =================================================
            # BARU FILTER OFFICIAL SETELAH DECODE
            # =================================================

            if not is_official_government_url(
                href
            ):

                continue


            collected.append({

                "url":
                    href,

                "search_text":
                    (
                        item.get(
                            "text"
                        )
                        or ""
                    ),

                "query":
                    query
            })


        print(
            "      decoded URLs:",
            decoded_count
        )


    # ========================================================
    # DEDUPE
    # ========================================================

    unique = []

    seen = set()


    for item in collected:

        url = (
            item[
                "url"
            ]
            .split("#")[0]
        )


        # buang tracking query umum
        if "?" in url:

            base = (
                url.split(
                    "?"
                )[0]
            )

        else:

            base = url


        key = (
            base.rstrip(
                "/"
            )
        )


        if key in seen:

            continue


        seen.add(
            key
        )


        item[
            "url"
        ] = url


        unique.append(
            item
        )


    print(
        "   Official URLs after filter:",
        len(
            unique
        )
    )


    for item in unique[:10]:

        print(
            "      +",
            item[
                "url"
            ]
        )


    return [

        item[
            "url"
        ]

        for item
        in unique[
            :MAX_OFFICIAL_PAGES_PER_TARGET
        ]
    ]

In [29]:
def extract_coordinates(
    content
):

    if not content:
        return []


    content = unquote(
        content
    )


    candidates = []


    # ========================================================
    # Google Maps:
    # @-3.123,114.123
    # ========================================================

    for match in re.finditer(
        r'@(-?\d{1,2}\.\d{4,}),'
        r'(-?\d{2,3}\.\d{4,})',
        content
    ):

        coord = valid_kalimantan_coordinate(
            match.group(1),
            match.group(2)
        )

        if coord:
            candidates.append(
                coord
            )


    # ========================================================
    # Google Maps embed:
    # !3dLAT!4dLON
    # ========================================================

    for match in re.finditer(
        r'!3d(-?\d{1,2}\.\d+)'
        r'!4d(-?\d{2,3}\.\d+)',
        content
    ):

        coord = valid_kalimantan_coordinate(
            match.group(1),
            match.group(2)
        )

        if coord:
            candidates.append(
                coord
            )


    # ========================================================
    # latitude / longitude JSON
    # ========================================================

    lat_matches = re.findall(
        r'["\']?latitude["\']?'
        r'\s*[:=]\s*["\']?'
        r'(-?\d+\.\d+)',
        content,
        flags=re.I
    )


    lon_matches = re.findall(
        r'["\']?longitude["\']?'
        r'\s*[:=]\s*["\']?'
        r'(-?\d+\.\d+)',
        content,
        flags=re.I
    )


    if (
        lat_matches
        and
        lon_matches
    ):

        for lat in lat_matches[:5]:

            for lon in lon_matches[:5]:

                coord = (
                    valid_kalimantan_coordinate(
                        lat,
                        lon
                    )
                )

                if coord:

                    candidates.append(
                        coord
                    )


    # ========================================================
    # OSM mlat / mlon
    # ========================================================

    mlat = re.search(
        r'mlat=(-?\d+\.\d+)',
        content,
        re.I
    )

    mlon = re.search(
        r'mlon=(-?\d+\.\d+)',
        content,
        re.I
    )


    if mlat and mlon:

        coord = (
            valid_kalimantan_coordinate(
                mlat.group(1),
                mlon.group(1)
            )
        )

        if coord:

            candidates.append(
                coord
            )


    # ========================================================
    # generic "lat, lon"
    # ========================================================

    for match in re.finditer(
        r'(?<!\d)'
        r'(-?[0-4]?\d\.\d{4,})'
        r'\s*[,;]\s*'
        r'(10[8-9]\.\d{4,}|'
        r'11\d\.\d{4,})',
        content
    ):

        coord = (
            valid_kalimantan_coordinate(
                match.group(1),
                match.group(2)
            )
        )

        if coord:

            candidates.append(
                coord
            )


    # ========================================================
    # DEDUPE
    # ========================================================

    seen = set()
    output = []


    for lat, lon in candidates:

        key = (
            round(
                lat,
                6
            ),
            round(
                lon,
                6
            )
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        output.append(
            (
                lat,
                lon
            )
        )


    return output

In [30]:
def extract_addresses(
    body_text,
    target
):

    if not body_text:
        return []


    area = (
        target[
            "area"
        ]
        .lower()
        .replace(
            "kabupaten ",
            ""
        )
        .replace(
            "kota ",
            ""
        )
    )


    province = (
        target[
            "province"
        ]
        .lower()
    )


    lines = [

        re.sub(
            r"\s+",
            " ",
            line
        ).strip()

        for line
        in body_text.splitlines()

        if line.strip()
    ]


    candidates = []


    for line in lines:

        low = (
            line.lower()
        )


        address_signal = (
            "alamat" in low
            or
            "jalan " in low
            or
            "jl." in low
            or
            "jl " in low
        )


        if not address_signal:
            continue


        if len(line) < 8:
            continue


        if len(line) > 350:
            continue


        score = 0


        if area in low:
            score += 5


        if province in low:
            score += 4


        if "alamat" in low:
            score += 4


        if (
            "jalan" in low
            or
            "jl." in low
        ):
            score += 3


        candidates.append(
            (
                score,
                line
            )
        )


    candidates.sort(
        reverse=True
    )


    unique = []

    seen = set()


    for score, address in candidates:

        key = (
            address.lower()
        )


        if key in seen:
            continue


        seen.add(
            key
        )

        unique.append(
            address
        )


    return unique[:8]

In [31]:
async def inspect_official_page(
    context,
    url,
    target
):

    page = (
        await context.new_page()
    )


    try:

        print(
            "      Inspect:",
            url
        )


        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=90_000
        )


        await page.wait_for_timeout(
            PAGE_WAIT_MS
        )


        final_url = (
            page.url
        )


        if not is_official_government_url(
            final_url
        ):

            return None


        title = (
            await page.title()
        )


        try:

            body_text = (
                await page.locator(
                    "body"
                ).inner_text(
                    timeout=5000
                )
            )

        except Exception:

            body_text = ""


        html = (
            await page.content()
        )


        # URL dari link / map / iframe
        urls = await page.locator(
            "[href], [src]"
        ).evaluate_all(
            """
            els => els.map(
                el =>
                    el.href ||
                    el.src ||
                    ''
            )
            """
        )


        combined = (
            html
            +
            "\n"
            +
            body_text
            +
            "\n"
            +
            "\n".join(
                urls
            )
        )


        coords = (
            extract_coordinates(
                combined
            )
        )


        addresses = (
            extract_addresses(
                body_text,
                target
            )
        )


        return {

            "url":
                final_url,

            "title":
                title,

            "coordinates":
                coords,

            "addresses":
                addresses
        }


    except Exception as e:

        print(
            "         Page error:",
            e
        )

        return None


    finally:

        await page.close()

In [32]:
async def geocode_address_playwright(
    context,
    address,
    target
):

    query = (
        f"{address}, "
        f"{target['area']}, "
        f"{target['province']}, "
        f"Indonesia"
    )


    url = (
        "https://nominatim.openstreetmap.org/"
        "search"
        "?format=jsonv2"
        "&limit=5"
        "&countrycodes=id"
        "&q="
        +
        quote_plus(
            query
        )
    )


    page = (
        await context.new_page()
    )


    try:

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60_000
        )


        await page.wait_for_timeout(
            800
        )


        body_text = (
            await page.locator(
                "body"
            ).inner_text()
        )


        data = json.loads(
            body_text
        )


        for result in data:

            lat = result.get(
                "lat"
            )

            lon = result.get(
                "lon"
            )


            coord = (
                valid_kalimantan_coordinate(
                    lat,
                    lon
                )
            )


            if not coord:
                continue


            return {

                "latitude":
                    coord[0],

                "longitude":
                    coord[1],

                "geocoder_query":
                    query,

                "geocoder_display_name":
                    result.get(
                        "display_name"
                    )
            }


        return None


    except Exception:

        return None


    finally:

        await page.close()

In [33]:
async def resolve_responder(
    context,
    search_page,
    target
):

    print(
        "\n============================================"
    )

    print(
        target[
            "target_id"
        ],
        "|",
        target[
            "name"
        ]
    )


    official_urls = (
        await search_official_pages_playwright(
            search_page,
            target
        )
    )


    print(
        "   Official pages:",
        len(
            official_urls
        )
    )


    for official_url in official_urls:

        page_data = (
            await inspect_official_page(
                context,
                official_url,
                target
            )
        )


        if not page_data:
            continue


        # ====================================================
        # BEST CASE:
        # coordinate langsung dari halaman pemerintah
        # ====================================================

        if page_data[
            "coordinates"
        ]:

            lat, lon = (
                page_data[
                    "coordinates"
                ][0]
            )


            print(
                "   ✓ official coordinate:",
                lat,
                lon
            )


            return {

                **target,

                "latitude":
                    lat,

                "longitude":
                    lon,

                "coordinate_quality":
                    "official_page_coordinate",

                "coordinate_method":
                    "extracted_from_official_page",

                "official_address":
                    (
                        page_data[
                            "addresses"
                        ][0]

                        if page_data[
                            "addresses"
                        ]

                        else None
                    ),

                "source_page_url":
                    page_data[
                        "url"
                    ],

                "source_page_title":
                    page_data[
                        "title"
                    ],

                "coordinate_source":
                    "official_government_page",

                "registry_quality":
                    "official"
            }


        # ====================================================
        # SECOND BEST:
        # official address -> Playwright Nominatim
        # ====================================================

        for address in (
            page_data[
                "addresses"
            ]
        ):

            geo = (
                await geocode_address_playwright(
                    context,
                    address,
                    target
                )
            )


            if not geo:
                continue


            print(
                "   ✓ official address geocoded:",
                geo[
                    "latitude"
                ],
                geo[
                    "longitude"
                ]
            )


            return {

                **target,

                "latitude":
                    geo[
                        "latitude"
                    ],

                "longitude":
                    geo[
                        "longitude"
                    ],

                "coordinate_quality":
                    "geocoded_official_address",

                "coordinate_method":
                    "official_address_playwright_geocode",

                "official_address":
                    address,

                "source_page_url":
                    page_data[
                        "url"
                    ],

                "source_page_title":
                    page_data[
                        "title"
                    ],

                "coordinate_source":
                    "official_government_address + Nominatim",

                "geocoder_display_name":
                    geo.get(
                        "geocoder_display_name"
                    ),

                "registry_quality":
                    "official_source_geocoded"
            }


    # ========================================================
    # NO CREDIBLE LOCATION FOUND
    # ========================================================

    print(
        "   ✗ not found"
    )


    return {

        **target,

        "latitude":
            None,

        "longitude":
            None,

        "coordinate_quality":
            "not_found",

        "coordinate_method":
            None,

        "official_address":
            None,

        "source_page_url":
            None,

        "source_page_title":
            None,

        "coordinate_source":
            None,

        "registry_quality":
            "unresolved"
    }

In [34]:
async def run_all_responders_playwright():

    # ========================================================
    # RESUME CHECKPOINT
    # ========================================================

    existing = {}


    if RESPONDER_AUDIT_OUT.exists():

        old = pd.read_csv(
            RESPONDER_AUDIT_OUT
        )


        for _, row in old.iterrows():

            existing[
                row[
                    "target_id"
                ]
            ] = row.to_dict()


        print(
            "Checkpoint loaded:",
            len(
                existing
            )
        )


    async with async_playwright() as p:

        browser = await p.chromium.launch(executable_path='C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe', 

            headless=HEADLESS,

            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )


        context = await browser.new_context(

            ignore_https_errors=True,

            viewport={
                "width": 1500,
                "height": 1000
            },

            user_agent=(
                "Mozilla/5.0 "
                "(X11; Linux x86_64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/131.0 Safari/537.36"
            )
        )


        search_page = (
            await context.new_page()
        )


        results = []


        for index, target in enumerate(
            TARGETS,
            start=1
        ):

            target_id = (
                target[
                    "target_id"
                ]
            )


            # ================================================
            # RESUME
            # ================================================

            if target_id in existing:

                old_result = (
                    existing[
                        target_id
                    ]
                )


                old_quality = str(
                    old_result.get(
                        "coordinate_quality",
                        ""
                    )
                )


                skip = (
                    old_quality
                    != "not_found"
                    or
                    not RETRY_NOT_FOUND
                )


                if skip:

                    print(
                        f"[{index}/{len(TARGETS)}] "
                        f"SKIP {target_id}"
                    )

                    results.append(
                        old_result
                    )

                    continue


            print(
                f"\n[{index}/{len(TARGETS)}]"
            )


            result = (
                await resolve_responder(
                    context,
                    search_page,
                    target
                )
            )


            results.append(
                result
            )


            # ================================================
            # CHECKPOINT SETIAP TARGET
            # ================================================

            checkpoint_df = (
                pd.DataFrame(
                    results
                )
            )


            checkpoint_df.to_csv(
                RESPONDER_AUDIT_OUT,
                index=False
            )


        await browser.close()


    return pd.DataFrame(
        results
    )

In [35]:
responders_df = (
    await run_all_responders_playwright()
)

NotImplementedError: 